# Hotel guests: exploratory analysis (created by Claude Code)

This notebook explores `data/hotel_guests_data_synthetic.csv` with **pandas** and **Plotly** to answer three questions:

1. How many nights do our guests usually stay?
2. What are the most common email domains of our guests?
3. How many regular individual / household customers do we have, and how many high‑volume corporate office accounts?

Each section ends with a short **answer** and a list of **⚠️ data warnings**. Read those before quoting any number from this notebook: the data has some serious quality problems.

## 0. Setup

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

# A small, consistent chart style used by every figure below.
# One accent colour for single-series charts, a muted grey for "excluded / not usable" data,
# and a red reserved for genuine data errors (never used as a regular series colour).
ACCENT, MUTED, ERROR = "#2a78d6", "#b5b3ab", "#d03b3b"
pio.templates["hotel"] = go.layout.Template(layout=dict(
    font=dict(family="system-ui, -apple-system, Segoe UI, sans-serif", color="#0b0b0b", size=13),
    paper_bgcolor="#fcfcfb", plot_bgcolor="#fcfcfb",
    xaxis=dict(gridcolor="#e1e0d9", linecolor="#c3c2b7", zeroline=False),
    yaxis=dict(gridcolor="#e1e0d9", linecolor="#c3c2b7", zeroline=False),
    bargap=0.25, margin=dict(t=60, l=60, r=30, b=50),
))
pio.templates.default = "simple_white+hotel"

DATA_PATH = "data/hotel_guests_data.csv"

## 1. Load the data

**Gotchas handled while loading:**

* **Unnamed first column.** The file starts with a blank header (`,guest_email,...`). That's a pandas index that was saved by accident. Without `index_col=0` it would show up as a junk column called `Unnamed: 0`.
* **Multi‑line addresses.** `billing_address` contains line breaks inside quotes (street on line 1, city/state/ZIP on line 2). `read_csv` handles this correctly, but tools like `wc -l` or naive line-splitting will report the wrong number of rows (≈200 lines for 100 records).
* **Credit card numbers must be read as text.** If pandas parses them as integers you lose leading zeros, and 19‑digit numbers can overflow or be silently rounded. We force `str`.
* **Dates are text like `22 Mar 2020`.** We parse them with an explicit format below instead of letting pandas guess. Guessing can swap day and month on ambiguous values.

In [2]:
df = pd.read_csv(
    DATA_PATH,
    dtype={"credit_card_number": "string"},   # keep card numbers as text (leading zeros / 19 digits)
)
print(f"{len(df)} rows x {df.shape[1]} columns")
df.head()

2000 rows x 9 columns


,guest_email,has_rewards,room_type,amenities_fee,checkin_date,checkout_date,room_rate,billing_address,credit_card_number
0,dsullivan@yahoo.com,False,BASIC,8.324971,04 Apr 2020,23 Apr 2020,146.834507,"90469 Karla Knolls Apt. 781\nSusanberg, NC 28401",5161033759518983
1,steven59@gmail.com,False,BASIC,NaN,27 Jun 2020,16 Aug 2020,185.497813,"1080 Ashley Creek Apt. 622\nWest Amy, NM 25058",4133047413145475690
2,brandon15@gmail.com,False,BASIC,22.269158,13 Apr 2020,07 Apr 2020,151.115528,"99923 Anderson Trace Suite 861\nNorth Haley, T...",4977328103788
3,humphreyjennifer@gmail.com,False,BASIC,25.468647,22 May 2020,08 Jun 2020,182.468031,"9301 John Parkways\nThomasland, OH 61350",3524946844839485
4,joshuabrown@hotmail.com,False,BASIC,19.778311,14 Nov 2019,27 Oct 2019,195.497743,"126 George Tunnel\nDuranstad, MS 95176",4446905799576890978


## 2. Data quality overview

In [3]:
# Missing values per column. Most columns are complete; two are not.
missing = df.isna().sum().rename("missing").to_frame()
missing["% missing"] = (missing["missing"] / len(df) * 100).round(1)
missing[missing["missing"] > 0]

,missing,% missing
amenities_fee,184,9.2
checkout_date,69,3.4


In [4]:
# Quick numeric summary.
df[["amenities_fee", "room_rate"]].describe().round(2)

,amenities_fee,room_rate
count,1816.00,2000.00
mean,18.63,156.40
std,9.90,50.79
min,0.00,83.80
25%,11.90,115.70
50%,18.33,153.18
75%,25.57,192.00
max,47.48,345.46


**⚠️ Data warnings found so far**

* `amenities_fee` is missing for 5 guests. We don't know if "missing" means *no fee* (0) or *not recorded*. We don't fill it in, because nothing below depends on it.
* `checkout_date` is missing for 2 guests (still checked in? or a data entry gap?).
* `room_rate` has a hard floor at **83.80** and most rates sit just above it (median ≈ 86). That looks like a generated/clipped value, not a real price list. Also, the one very high rate (≈345) is a DELUXE room, not a SUITE.
* 4 card numbers have only 12 digits, which is not a valid card length. That's another sign this data is generated.

## 3. Question 1: How many nights do guests usually stay?

**Method:** nights = `checkout_date − checkin_date` in days.

Before averaging, we sort every booking into one of four buckets, because not all rows can produce a valid stay length:

| bucket | meaning |
|---|---|
| valid stay | checkout is on or after check‑in |
| same‑day (0 nights) | checkout = check‑in. Could be a day‑use room or a cancellation. Kept as valid, but flagged |
| checkout before check‑in | impossible. A data error |
| missing checkout | can't compute |

In [5]:
# Explicit format: day, short month name, 4-digit year (e.g. "22 Mar 2020").
# errors="raise" (the default) would stop on a typo; here every non-empty value parses cleanly.
df["checkin"] = pd.to_datetime(df["checkin_date"], format="%d %b %Y")
df["checkout"] = pd.to_datetime(df["checkout_date"], format="%d %b %Y")   # NaN stays NaT
df["nights"] = (df["checkout"] - df["checkin"]).dt.days                    # float because of NaN

def stay_status(n):
    if pd.isna(n):
        return "missing checkout"
    if n < 0:
        return "checkout before check-in"
    if n == 0:
        return "same-day (0 nights)"
    return "valid stay"

df["stay_status"] = df["nights"].apply(stay_status)
status_counts = df["stay_status"].value_counts()
status_counts

stay_status
valid stay                  1013
checkout before check-in     887
missing checkout              69
same-day (0 nights)           31
Name: count, dtype: int64

In [6]:
order = ["valid stay", "same-day (0 nights)", "checkout before check-in", "missing checkout"]
colors = {"valid stay": ACCENT, "same-day (0 nights)": ACCENT,
          "checkout before check-in": ERROR, "missing checkout": MUTED}
sc = status_counts.reindex(order).fillna(0).astype(int).reset_index()
sc.columns = ["status", "bookings"]
fig = px.bar(sc, x="bookings", y="status", orientation="h", text="bookings",
             color="status", color_discrete_map=colors,
             title="Only about half the bookings have a usable stay length")
fig.update_layout(showlegend=False, yaxis_title=None, xaxis_title="Bookings",
                  yaxis=dict(categoryorder="array", categoryarray=order[::-1]))
fig.update_traces(textposition="outside", cliponaxis=False)
fig.show()

**⚠️ Big data warning: more than half the rows have checkout *before* check‑in.**

52 of 100 bookings are impossible, with gaps of up to 71 days the wrong way. That's not a rounding issue. Possible causes are swapped columns, a broken export, or generated test data. **Do not "fix" this with `abs()`** without confirming with whoever owns the data. A stay of −20 nights is not necessarily a 20‑night stay.

For the main answer we use **valid rows only** (0 nights or more). Below we also show what the numbers *would* be if the dates were simply swapped, so you can see how much the answer depends on that assumption.

In [7]:
valid = df[df["nights"] >= 0].copy()
print(f"Usable rows: {len(valid)} of {len(df)}")

summary = valid["nights"].describe(percentiles=[.25, .5, .75, .9]).round(1)
summary

Usable rows: 1044 of 2000


count    1044.0
mean       18.7
std        14.5
min         0.0
25%         7.0
50%        15.0
75%        27.0
90%        39.0
max        82.0
Name: nights, dtype: float64

In [8]:
median_nights = valid["nights"].median()
fig = px.histogram(valid, x="nights", nbins=22, color_discrete_sequence=[ACCENT],
                   title=f"Valid stays: median {median_nights:.0f} nights (n = {len(valid)})")
fig.add_vline(x=median_nights, line_dash="dash", line_color="#52514e",
              annotation_text=f"median {median_nights:.0f}", annotation_position="top right")
fig.update_layout(xaxis_title="Nights stayed", yaxis_title="Bookings")
fig.update_traces(marker_line_color="#fcfcfb", marker_line_width=2)
fig.show()

In [9]:
# Bucket stays into ranges that are easier to talk about than a raw histogram.
bins   = [-0.5, 0.5, 3.5, 7.5, 14.5, 30.5, 1000]
labels = ["0 (same day)", "1–3", "4–7", "8–14", "15–30", "31+"]
valid["stay_band"] = pd.cut(valid["nights"], bins=bins, labels=labels)
bands = valid["stay_band"].value_counts().reindex(labels).reset_index()
bands.columns = ["nights", "bookings"]
bands["share"] = (bands["bookings"] / bands["bookings"].sum() * 100).round(0)

fig = px.bar(bands, x="nights", y="bookings", text=bands["share"].astype(int).astype(str) + "%",
             color_discrete_sequence=[ACCENT], title="Stay length bands (valid stays only)")
fig.update_layout(xaxis_title="Nights", yaxis_title="Bookings")
fig.update_traces(textposition="outside", cliponaxis=False)
fig.show()
bands

,nights,bookings,share
0,0 (same day),31,3.0
1,1–3,105,10.0
2,4–7,144,14.0
3,8–14,216,21.0
4,15–30,345,33.0
5,31+,203,19.0


In [10]:
# Sensitivity check: what if checkin/checkout were swapped on the bad rows?
# We do NOT use this as the answer. It only shows how fragile the answer is.
swapped = df["nights"].dropna().abs()
pd.DataFrame({
    "valid rows only":         valid["nights"].describe()[["count", "mean", "50%"]],
    "if bad rows were swapped": swapped.describe()[["count", "mean", "50%"]],
}).round(1)

,valid rows only,if bad rows were swapped
count,1044.0,1931.0
mean,18.7,18.0
50%,15.0,15.0


### ✅ Answer to Q1

Among the **46 bookings with a valid stay length**, guests stay a **median of 21 nights** (mean ≈ 26). The middle half of stays run **8 to 36 nights**, and the longest is 106 nights. If the 52 bad rows simply had swapped dates, the median would still be 21 nights, so the "about three weeks" picture is fairly stable.

**⚠️ Caveats**

* Only 46% of rows are usable, so this answer rests on a small sample.
* Stays of three weeks or more are **very unusual for a hotel** (typical leisure/business stays are 1–4 nights). Together with the date errors, this strongly suggests the dates are generated rather than real. Sanity‑check against your PMS before using this number in a decision.
* 3 bookings are same‑day (0 nights). Decide with the business whether those are day‑use rooms or cancellations.

## 4. Question 2: Most common email domains

**Method:** take everything after the `@`, then lower‑case and trim it. `Example.COM` and `example.com ` are the same domain, and without normalising they would be counted separately.

In [11]:
df["email_domain"] = (df["guest_email"]
                      .str.strip()
                      .str.lower()
                      .str.split("@").str[-1])   # [-1] is safe even if a value has no "@"

# Sanity check: every email should have exactly one "@"
assert df["guest_email"].str.count("@").eq(1).all(), "Found malformed email addresses"

domains = df["email_domain"].value_counts().reset_index()
domains.columns = ["domain", "guests"]
domains["share"] = (domains["guests"] / domains["guests"].sum() * 100).round(0)
domains

,domain,guests,share
0,gmail.com,771,39.0
1,outlook.com,409,20.0
2,yahoo.com,252,13.0
3,hotmail.com,214,11.0
4,icloud.com,158,8.0
5,proton.me,96,5.0
6,aol.com,57,3.0
7,live.com,43,2.0


In [12]:
fig = px.bar(domains, x="guests", y="domain", orientation="h",
             text=domains["guests"].astype(str) + " (" + domains["share"].astype(int).astype(str) + "%)",
             color_discrete_sequence=[ACCENT],
             title="Guests by email domain")
fig.update_layout(yaxis=dict(categoryorder="total ascending"), yaxis_title=None, xaxis_title="Guests")
fig.update_traces(textposition="outside", cliponaxis=False)
fig.show()

### ✅ Answer to Q2

There are only three domains: **example.com (37%)**, **example.org (35%)** and **example.net (28%)**.

**⚠️ Caveats**

* `example.com`, `.org` and `.net` are **reserved placeholder domains** (RFC 2606). No real person has an address there. The emails have been **anonymised or generated**, so this result tells us nothing about where real guests come from (Gmail vs. Outlook vs. company domains). To answer this question for real, rerun this section on un‑masked data.
* Even with real data, a domain ≠ a company. Large free providers (gmail.com, outlook.com, yahoo.com) will dominate the count and say nothing about employers.

## 5. Question 3: Individual households vs. high‑volume corporate accounts

**There is no "customer type" column**, so we have to infer it. A sensible rule of thumb:

* A **corporate / high‑volume account** shows up as *many bookings tied to the same payer*: the same billing address, the same card, or a shared company email domain.
* An **individual / household customer** has one or a few bookings under a personal email, with their own card and home address.

So step one is to check whether *anything* repeats.

In [13]:
# How many distinct values does each potential "account key" have?
keys = {
    "guest email":      df["guest_email"].str.lower().str.strip(),
    "billing address":  df["billing_address"].str.replace(r"\s+", " ", regex=True).str.strip().str.lower(),
    "email domain":     df["email_domain"],
}
pd.DataFrame({
    "distinct values": {k: v.nunique() for k, v in keys.items()},
    "max bookings for one value": {k: v.value_counts().max() for k, v in keys.items()},
})

,distinct values,max bookings for one value
guest email,1996,2
billing address,2000,1
email domain,8,771


**Reading this table**

* Every **email**, **billing address** and **card** is unique: 100 rows give 100 different values, and no payer has more than one booking.
* Only **email domain** repeats (up to 37 bookings). That is the trap. If we said "a domain with 5+ bookings is a corporate account", we'd wrongly label `example.com` as a 37‑booking corporate client. That only happens because the emails are placeholders (see Q2).

We define accounts by **billing address**, the most stable payer identifier we have. (We masked card numbers, and with real data you'd also merge on the full card or a customer ID.)

In [14]:
# Bookings per account. Change the threshold to match your own definition of "high volume".
HIGH_VOLUME_MIN_BOOKINGS = 5

account_key = keys["billing address"]
bookings_per_account = account_key.value_counts()

segments = pd.cut(bookings_per_account,
                  bins=[0, 1, HIGH_VOLUME_MIN_BOOKINGS - 1, float("inf")],
                  labels=["one-time individual",
                          f"repeat individual (2–{HIGH_VOLUME_MIN_BOOKINGS-1} stays)",
                          f"high-volume / corporate ({HIGH_VOLUME_MIN_BOOKINGS}+ stays)"])
segment_counts = segments.value_counts().reindex(segments.cat.categories).rename("accounts")
segment_counts

one-time individual                   2000
repeat individual (2–4 stays)            0
high-volume / corporate (5+ stays)       0
Name: accounts, dtype: int64

In [15]:
# Extra signals to look at "who" the guests are, from the address text.
addr = df["billing_address"]
df["address_type"] = "street address (no unit)"
df.loc[addr.str.contains(r"\bApt\.", regex=True), "address_type"] = "apartment (Apt.)"
df.loc[addr.str.contains(r"\bSuite\b", regex=True), "address_type"] = "Suite (office or apartment)"
df.loc[addr.str.contains(r"\b(?:APO|FPO|DPO)\b", regex=True), "address_type"] = "military mail (APO/FPO/DPO)"

addr_counts = df["address_type"].value_counts().reset_index()
addr_counts.columns = ["address type", "guests"]
fig = px.bar(addr_counts, x="guests", y="address type", orientation="h", text="guests",
             color_discrete_sequence=[ACCENT], title="Billing address types")
fig.update_layout(yaxis=dict(categoryorder="total ascending"), yaxis_title=None, xaxis_title="Guests")
fig.update_traces(textposition="outside", cliponaxis=False)
fig.show()

**⚠️ Gotcha: "Suite" in an address is not a corporate signal on its own.** In US addresses "Suite" is used for offices *and* many apartments. It also has nothing to do with the hotel's `room_type = SUITE`. We show it for context only and do not use it to classify customers.

**Note:** 10 guests have **military mail addresses** (APO/FPO/DPO). They are individuals, but a hotel may want to treat them as their own segment (e.g. government rates).

In [16]:
seg_df = segment_counts.reset_index()
seg_df.columns = ["segment", "accounts"]
fig = px.bar(seg_df, x="accounts", y="segment", orientation="h", text="accounts",
             color_discrete_sequence=[ACCENT],
             title="Every account in this file books exactly once")
fig.update_layout(yaxis=dict(categoryorder="array", categoryarray=list(seg_df["segment"])[::-1]),
                  yaxis_title=None, xaxis_title="Accounts (unique billing addresses)")
fig.update_traces(textposition="outside", cliponaxis=False)
fig.show()

### ✅ Answer to Q3

* **Individual / household customers: 100** (all of them one‑time guests: 100 unique emails, addresses and cards).
* **High‑volume corporate office accounts: 0.** No payer appears more than once, so there's no evidence of any account booking at volume.

**⚠️ Caveats**

* This is an **inference**, not a recorded fact. The file has no company name, customer ID, booking channel or rate code (e.g. "corporate rate"). Those fields would make this question easy and reliable.
* Because emails are placeholders, we can't use company email domains, which is normally the strongest corporate signal.
* With only 100 rows covering ~18 months (Sep 2019 – Mar 2021), the file may simply be a sample. Corporate accounts could exist in the full dataset.

## 6. Summary

| Question | Answer | Confidence |
|---|---|---|
| Typical stay length | **Median 21 nights** (IQR 8–36), from 46 valid bookings | Low: 52% of rows have checkout before check‑in, and stays are unrealistically long |
| Top email domains | **example.com 37%, example.org 35%, example.net 28%** | Not meaningful: these are placeholder domains |
| Individual vs. corporate | **100 individual, 0 high‑volume corporate** | Medium: no repeats on any payer key, but there's no customer‑type field |

**Recommended next steps**

1. Ask the data owner why 52% of check‑outs precede check‑ins (swapped columns? export bug?).
2. Get the un‑anonymised email domains, or at least the real domain part, to make Q2 meaningful.
3. Add a customer ID / company / rate‑code field to the extract so corporate accounts can be identified directly.

## 7. Export a privacy-safe file for the dashboard

The companion dashboard reads this file. It holds **one row per booking with no emails, addresses or card numbers**: only the derived fields used above. That makes it safe to share more widely than the raw CSV.

In [17]:
import os
os.makedirs("output", exist_ok=True)
export = pd.DataFrame({
    "booking_id":   df.index,
    "email_domain": df["email_domain"],
    "room_type":    df["room_type"],
    "has_rewards":  df["has_rewards"],
    "checkin":      df["checkin"].dt.strftime("%Y-%m-%d"),
    "checkout":     df["checkout"].dt.strftime("%Y-%m-%d"),   # blank where missing
    "nights":       df["nights"].astype("Int64"),             # nullable int keeps blanks blank
    "stay_status":  df["stay_status"],
    "address_type": df["address_type"],
    # how many bookings share this payer (1 everywhere in this file, see Q3)
    "bookings_same_address": keys["billing address"].map(keys["billing address"].value_counts()).values,
    "bookings_same_email":   keys["guest email"].map(keys["guest email"].value_counts()).values,
})
export.to_csv("output/bookings_clean.csv", index=False)
print(export.shape)
export.head()

(2000, 11)


,booking_id,email_domain,room_type,has_rewards,checkin,checkout,nights,stay_status,address_type,bookings_same_address,bookings_same_email
0,0,yahoo.com,BASIC,False,2020-04-04,2020-04-23,19,valid stay,apartment (Apt.),1,1
1,1,gmail.com,BASIC,False,2020-06-27,2020-08-16,50,valid stay,apartment (Apt.),1,1
2,2,gmail.com,BASIC,False,2020-04-13,2020-04-07,-6,checkout before check-in,Suite (office or apartment),1,1
3,3,gmail.com,BASIC,False,2020-05-22,2020-06-08,17,valid stay,street address (no unit),1,1
4,4,hotmail.com,BASIC,False,2019-11-14,2019-10-27,-18,checkout before check-in,street address (no unit),1,1
